# CyberPulse AI: Wi-Fi Device Classification and Anomaly Detection

**CyberPulse AI** is an enhanced machine learning project for classifying Wi-Fi devices as 'Cisco Systems Inc.' or not, using signal strength, channel, and SSID data. It includes advanced features for **network penetration testing**, such as anomaly detection for rogue access points and model comparison for robust device fingerprinting.

## Objectives
- Classify Wi-Fi devices with improved accuracy using KNN and Random Forest.
- Analyze signal trends to support network diagnostics.
- Detect potential rogue access points via anomaly detection.
- Provide visualizations and exportable models for penetration testing workflows.

## Dataset
- **File**: `wifi_data.csv`
- **Columns**: Time, MAC Address, Vendor, SSID, Signal Strength, Channel, Survey

## Enhancements
- Hyperparameter tuning for KNN.
- Random Forest classifier for comparison.
- Feature engineering: SSID length.
- Anomaly detection for rogue APs.
- Robust data cleaning and error handling.
- Enhanced visualizations and model export.

## Import Libraries

Import required libraries for data processing, modeling, visualization, and model persistence.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, classification_report
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import IsolationForest
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

## Load and Clean Dataset

Load the Wi-Fi dataset and perform data cleaning to handle missing or invalid values.

In [ ]:
try:
    wifi_data = pd.read_csv('wifi_data.csv', encoding='ISO-8859-1', sep=';')
except FileNotFoundError:
    print('Error: wifi_data.csv not found. Please ensure the file is in the project directory.')
    raise
except pd.errors.ParserError:
    print('Error: Invalid CSV format. Ensure the file uses semicolon delimiters.')
    raise

# Data cleaning
wifi_data.dropna(subset=['Time', 'Vendor', 'Signal.Strength', 'Channel', 'SSID'], inplace=True)
wifi_data['Time'] = pd.to_datetime(wifi_data['Time'], errors='coerce')
wifi_data = wifi_data.dropna(subset=['Time']).sort_values('Time')
wifi_data['Channel'] = pd.to_numeric(wifi_data['Channel'], errors='coerce')
wifi_data = wifi_data.dropna(subset=['Channel'])
wifi_data['Signal.Strength'] = pd.to_numeric(wifi_data['Signal.Strength'], errors='coerce')
wifi_data = wifi_data.dropna(subset=['Signal.Strength'])

print('Dataset shape after cleaning:', wifi_data.shape)
wifi_data.head()

## Feature Engineering

Create a new feature: SSID length, which may capture patterns in Wi-Fi network names.

In [ ]:
wifi_data['SSID_Length'] = wifi_data['SSID'].apply(lambda x: len(str(x)))

# Define target and features
y_Vendor = (wifi_data['Vendor'] == 'Cisco Systems Inc.').astype(int)
X = wifi_data[['Signal.Strength', 'Channel', 'SSID_Length']]

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y_Vendor, test_size=0.3, random_state=42, stratify=y_Vendor)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Train and Tune KNN Model

Use GridSearchCV to find the optimal number of neighbors for KNN.

In [ ]:
knn = KNeighborsClassifier()
param_grid = {'n_neighbors': [3, 5, 7, 9, 11]}
grid_search = GridSearchCV(knn, param_grid, cv=5, scoring='f1', n_jobs=-1)
grid_search.fit(X_train_scaled, y_train)

best_knn = grid_search.best_estimator_
print('Best KNN Parameters:', grid_search.best_params_)
print('Best F1-Score (CV):', grid_search.best_score_)

# Evaluate KNN
y_pred_knn = best_knn.predict(X_test_scaled)
print('\nKNN Classification Report:\n', classification_report(y_test, y_pred_knn, target_names=['Not Cisco', 'Cisco']))

# Save KNN model
joblib.dump(best_knn, 'knn_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

## Train Random Forest Model

Train a Random Forest classifier for comparison with KNN.

In [ ]:
rf = RandomForestClassifier(random_state=42, n_jobs=-1)
rf.fit(X_train_scaled, y_train)

# Evaluate Random Forest
y_pred_rf = rf.predict(X_test_scaled)
print('\nRandom Forest Classification Report:\n', classification_report(y_test, y_pred_rf, target_names=['Not Cisco', 'Cisco']))

# Save Random Forest model
joblib.dump(rf, 'rf_model.pkl')

## Compare Model Performance

Visualize the performance of KNN and Random Forest using a bar plot.

In [ ]:
def plot_model_comparison(y_test, y_pred_knn, y_pred_rf):
    metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
    knn_scores = [accuracy_score(y_test, y_pred_knn), precision_score(y_test, y_pred_knn), 
                  recall_score(y_test, y_pred_knn), f1_score(y_test, y_pred_knn)]
    rf_scores = [accuracy_score(y_test, y_pred_rf), precision_score(y_test, y_pred_rf), 
                 recall_score(y_test, y_pred_rf), f1_score(y_test, y_pred_rf)]
    
    x = np.arange(len(metrics))
    width = 0.35
    
    plt.figure(figsize=(10, 6))
    plt.bar(x - width/2, knn_scores, width, label='KNN', color='skyblue')
    plt.bar(x + width/2, rf_scores, width, label='Random Forest', color='lightcoral')
    plt.xlabel('Metrics')
    plt.ylabel('Score')
    plt.title('KNN vs Random Forest Performance')
    plt.xticks(x, metrics)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig('model_comparison.png')
    plt.show()

plot_model_comparison(y_test, y_pred_knn, y_pred_rf)

## Anomaly Detection

Use Isolation Forest to detect potential rogue access points based on signal strength and channel outliers.

In [ ]:
iso_forest = IsolationForest(contamination=0.05, random_state=42)
anomalies = iso_forest.fit_predict(X[['Signal.Strength', 'Channel']])
wifi_data['Anomaly'] = anomalies

# Plot anomalies
plt.figure(figsize=(10, 6))
plt.scatter(wifi_data[wifi_data['Anomaly'] == 1]['Channel'], 
            wifi_data[wifi_data['Anomaly'] == 1]['Signal.Strength'], 
            color='blue', label='Normal', alpha=0.6)
plt.scatter(wifi_data[wifi_data['Anomaly'] == -1]['Channel'], 
            wifi_data[wifi_data['Anomaly'] == -1]['Signal.Strength'], 
            color='red', label='Anomaly (Potential Rogue AP)', alpha=0.6)
plt.title('Anomaly Detection: Signal Strength vs Channel')
plt.xlabel('Channel')
plt.ylabel('Signal Strength (dBm)')
plt.legend()
plt.grid(True)
plt.savefig('anomaly_detection.png')
plt.show()

print('Number of detected anomalies (potential rogue APs):', sum(anomalies == -1))

## Signal Strength by Vendor

Visualize signal strength distribution by vendor using a box plot.

In [ ]:
# Create a binary vendor class so both Cisco and non-Cisco groups are compared.
# (The previous version filtered on a non-existent 'Other' label, leaving only one group.)
wifi_data['Vendor_Class'] = np.where(wifi_data['Vendor'] == 'Cisco Systems Inc.', 'Cisco', 'Other')

plt.figure(figsize=(8, 6))
sns.boxplot(x='Vendor_Class', y='Signal.Strength', data=wifi_data, hue='Vendor_Class', palette='Set2', legend=False)
plt.title('Signal Strength Distribution by Vendor')
plt.xlabel('Vendor')
plt.ylabel('Signal Strength (dBm)')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.savefig('signal_strength_by_vendor.png')
plt.show()

## Signal Strength vs Channel

Plot signal strength vs channel with a regression line.

In [ ]:
def plot_signal_strength_vs_channel(data):
    plt.figure(figsize=(8, 6))
    plt.scatter(data['Channel'], data['Signal.Strength'], color='blue', alpha=0.6)
    
    X_channel = data['Channel'].values.reshape(-1, 1)
    y_strength = data['Signal.Strength'].values
    lin_reg = LinearRegression()
    lin_reg.fit(X_channel, y_strength)
    y_pred = lin_reg.predict(X_channel)
    plt.plot(data['Channel'], y_pred, color='red', linewidth=2, label='Linear Regression')
    
    plt.title('Signal Strength vs Channel with Regression Line')
    plt.xlabel('Channel')
    plt.ylabel('Signal Strength (dBm)')
    plt.ylim(-100, -20)
    plt.grid(True)
    plt.legend()
    plt.savefig('signal_strength_vs_channel_with_regression.png')
    plt.show()

plot_signal_strength_vs_channel(wifi_data)

## Wi-Fi Info Table

Display a sample of the dataset as a table.

In [ ]:
def plot_wifi_info_table(data):
    columns_to_display = ['Time', 'MAC.Address', 'Vendor', 'SSID', 'Anomaly']
    table_data = data[columns_to_display].head()
    
    fig, ax = plt.subplots(figsize=(10, 2))
    ax.axis('off')
    table = ax.table(cellText=table_data.values, colLabels=table_data.columns, cellLoc='center', loc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.2, 1.2)
    
    plt.savefig('wifi_info_table.png')
    plt.show()

plot_wifi_info_table(wifi_data)

## Signal Strength Over Time

Plot signal strength over time with a rolling average.

In [ ]:
def plot_signal_strength_over_time(data):
    plt.figure(figsize=(10, 6))
    plt.plot(data['Time'], data['Signal.Strength'], label='Signal Strength', color='blue')
    
    if len(data) >= 3:
        rolling_avg = data['Signal.Strength'].rolling(window=3).mean()
        plt.plot(data['Time'], rolling_avg, label='Rolling Average (window=3)', color='orange')
    
    plt.title('Signal Strength Over Time')
    plt.xlabel('Time')
    plt.ylabel('Signal Strength (dBm)')
    plt.grid(True)
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig('signal_strength_over_time.png')
    plt.show()

plot_signal_strength_over_time(wifi_data)

## Signal Strength vs Time Difference

Plot signal strength vs time difference with a regression line.

In [ ]:
time_diff = (wifi_data['Time'] - wifi_data['Time'].min()).dt.total_seconds()

def plot_signal_strength_vs_time_diff(time_diff, data):
    plt.figure(figsize=(8, 6))
    plt.scatter(time_diff, data['Signal.Strength'], color='blue', alpha=0.6)
    
    X_time = time_diff.values.reshape(-1, 1)
    y_strength = data['Signal.Strength'].values
    lin_reg_time = LinearRegression()
    lin_reg_time.fit(X_time, y_strength)
    y_pred_time = lin_reg_time.predict(X_time)
    plt.plot(time_diff, y_pred_time, color='red', linewidth=2, label='Linear Regression')
    
    plt.title('Signal Strength vs Time Difference (Seconds)')
    plt.xlabel('Time Difference (Seconds)')
    plt.ylabel('Signal Strength (dBm)')
    plt.grid(True)
    plt.legend()
    plt.savefig('signal_strength_vs_time_diff.png')
    plt.show()

plot_signal_strength_vs_time_diff(time_diff, wifi_data)

## Penetration Testing Applications

This project supports network penetration testing by:
- **Device Fingerprinting**: Accurate classification (KNN, Random Forest) identifies Cisco devices for targeted vulnerability scans.
- **Rogue AP Detection**: Anomaly detection flags unusual signal patterns, aiding in identifying unauthorized access points.
- **Network Mapping**: Vendor classification and signal analysis help map network topology.
- **Toolkit Integration**: Exported models (`knn_model.pkl`, `rf_model.pkl`) can be integrated into real-time pentesting tools.